# Dock 05

## Configuration

In [1]:
#pull in local configuration
%run config.py
!cat config.py

GNINA_LOC = "gnina"
GNINA_PARAMETER = ""


In [2]:
sort_by_cnnvs = True

In [3]:
ligdictjson = "preprocessed/ligand.json"
prodictjson = "preprocessed/protein.json"

prodir = "preprocessed/proteinprep/"
dockeddir = "preprocessed/dockedligands/"
decoydir = "preprocessed/decoys/"
idealdir = "preprocessed/minimized_ideal_ligand/stillypuy/"
#idealdir = "preprocessed/minimized_ideal_ligand/"
resultsdir = "results/"
decoy = "generated_decoys_activeER.sdf"
decoy = "generated_decoys_activeER_filtered_FINAL.sdf"
docked = resultsdir + "docked.sdf"
log = resultsdir + "gninalog.txt"
rmsdlog = resultsdir + "rmsd.txt"
results = resultsdir + "results.csv"

In [4]:
sourceproteinprocesseddir = "../proteinprep01/chemfiles/"
sourceligandidealdir = "../mfaber_workflow/ER_Ligand_Prep/Ideal_Ligand/"
sourceligandidealdir = "../mfaber_workflow/ER_Ligand_Prep/minimized_ideal_ligand/"
localmayaprocesseddir = "preprocessed/maya_preprocessed_ligands/"

In [5]:
proteindict = {}
dockedliganddict = {}
idealliganddict = {}
gninaoptdict = {}
prepdict = {}
decoydf = None

In [6]:
import copy
def reportdict(rows, columns):
    lines = []
    if len(rows) == 0:
        return (lines) 
    inner_keys = set()
    for r in rows.values():
        inner_keys.update(r.keys())
    col_widths = {}
    col_widths["id"] = max(len("id"), max(len(k) for k in rows))
    for col in inner_keys:
        header_len = len(col)
        data_len = max(len(str(r.get(col, ""))) for r in rows.values())
        col_widths[col] = max(header_len, data_len)
    header = "  ".join(f"{col:<{col_widths[col]}}" for col in columns)
    lines.append(header)
    for outer_key, inner in rows.items():
        cells = [f"{outer_key:<{col_widths['id']}}"]
        for col in columns[1:]:
            cells.append(f"{str(inner.get(col, '')):<{col_widths[col]}}")
        lines.append("  ".join(cells))
    return (lines)

In [7]:
import time

def format_time(seconds):
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    return f"{hours:02d}:{minutes:02d}:{secs:02d}"

In [ ]:
#copy preprocessed proteins and docked ligand files
# This code only needs to be executed once assuming no new protein processing has taken place.
!cp -R {sourceproteinprocesseddir}* {localprocesseddir}
!ls -al {localprocesseddir}

In [ ]:
# Copy ideal ligand files that were processed by Maya if necessary.
# This block needs to be run only once assuming now more ligands have been processed by Maya.
# Maya processed these ligands and we will pull a copy into this version of Dock.
# Let's copy into a directory that provides a clue where the files came from.
#!cp -R {sourceligandidealdir} {localprocesseddir}
!cp -R {sourceligandidealdir}* {localmayaprocesseddir}

## Load and process decoy ligands from .sdf

In [ ]:
import pandas as pd
from rdkit.Chem import PandasTools

sdf_path = decoydir + decoy

decoydf = PandasTools.LoadSDF(
    sdf_path,
    molColName="ROMol",
    smilesName="SMILES",
    includeFingerprints=False,
    removeHs=False,
    strictParsing=True
)

# Add numbered ID column
decoydf["ID"] = [f"decoy{i+1}" for i in range(len(decoydf))]

print(decoydf.head())
print(decoydf.columns)
print(decoydf.shape)     # (rows, columns)
print(decoydf.info(memory_usage='deep'))

In [ ]:
from openbabel import openbabel as ob
from rdkit import Chem

def writeonedecoytoFS(row):
    
    # Your RDKit mol
    rdkit_mol = row['ROMol']

    # Convert RDKit MolBlock → Open Babel OBMol
    molblock = Chem.MolToMolBlock(rdkit_mol)
    ob_mol = ob.OBMol()
    ob_conversion = ob.OBConversion()
    ob_conversion.SetInFormat("mdl")  # MOL block format
    ob_conversion.ReadString(ob_mol, molblock)

    # Write out GNINA-ready SDF
    ob_conversion.SetOutFormat("sdf")
    sdf_string = ob_conversion.WriteString(ob_mol)

    with open(decoydir + "decoyligand.sdf", "w") as f:
        f.write(sdf_string)

## Populate docked ligand dictionary

In [8]:
dockedliganddict = {}

In [9]:
import json

# Read from text file
with open(ligdictjson, "r") as f:
    dockedliganddict = json.load(f)

# Which docked ligands are available?
lines = reportdict(dockedliganddict, ["id","source","prep","localfilename"])
print("\n".join(lines))

id               source    prep          localfilename        
EST_redock_1ERE  1ERE.pdb  rdkit_save_H  EST_redock_1ERE_A.sdf
EST_redock_1GWR  1GWR.pdb  rdkit_save_H  EST_redock_1GWR_A.sdf
EST_redock_3UUD  3UUD.pdb  rdkit_save_H  EST_redock_3UUD_A.sdf
EST_redock_6CBZ  6CBZ.pdb  rdkit_save_H  EST_redock_6CBZ_A.sdf
DES_redock_3ERD  3ERD.pdb  rdkit_save_H  DES_redock_3ERD_A.sdf
DES_redock_4ZN7  4ZN7.pdb  rdkit_save_H  DES_redock_4ZN7_A.sdf
27M_redock_4MGC  4MGC.pdb  rdkit_save_H  27M_redock_4MGC_A.sdf
27J_redock_4MG8  4MG8.pdb  rdkit_save_H  27J_redock_4MG8_A.sdf
36J_redock_4TUZ  4TUZ.pdb  rdkit_save_H  36J_redock_4TUZ_A.sdf
2OH_redock_3UU7  3UU7.pdb  rdkit_save_H  2OH_redock_3UU7_A.sdf
27K_redock_4MG9  4MG9.pdb  rdkit_save_H  27K_redock_4MG9_A.sdf
27L_redock_4MGA  4MGA.pdb  rdkit_save_H  27L_redock_4MGA_A.sdf
EST_redock_1G50  1G50.pdb  rdkit_save_H  EST_redock_1G50_A.sdf


## Populate protein dictionary

In [10]:
import json

# Read from text file
with open(prodictjson, "r") as f:
    proteindict = json.load(f)    

lines = reportdict(proteindict, ["id","name","prep","localfilename_fixed"])
print("\n".join(lines))

id    name  prep          localfilename_fixed
1ERE        PDBfixfunc74  1ERE_A_fixed.pdb   
1GWR        PDBfixfunc74  1GWR_A_fixed.pdb   
3UUD        PDBfixfunc74  3UUD_A_fixed.pdb   
6CBZ        PDBfixfunc74  6CBZ_A_fixed.pdb   
3ERD        PDBfixfunc74  3ERD_A_fixed.pdb   
4ZN7        PDBfixfunc74  4ZN7_A_fixed.pdb   
4MGC        PDBfixfunc74  4MGC_A_fixed.pdb   
4MG8        PDBfixfunc74  4MG8_A_fixed.pdb   
4TUZ        PDBfixfunc74  4TUZ_A_fixed.pdb   
3UU7        PDBfixfunc74  3UU7_A_fixed.pdb   
4MG9        PDBfixfunc74  4MG9_A_fixed.pdb   
4MGA        PDBfixfunc74  4MGA_A_fixed.pdb   
1G50        PDBfixfunc74  1G50_A_fixed.pdb   


In [11]:
# Add docked ligand id into protein dictionary. Next iteration this will happen in the 'pre process protein' code.

for (k1, v1), (k2, v2) in zip(proteindict.items(), dockedliganddict.items()):
    #print(f"Key: {k1}, Dict1: {v1}, Dict2: {v2}")
    v1['dockedid'] = v2['id']

lines = reportdict(proteindict, ["id","name","prep","localfilename_fixed","dockedid"])
print("\n".join(lines))

id    name  prep          localfilename_fixed  dockedid       
1ERE        PDBfixfunc74  1ERE_A_fixed.pdb     EST_redock_1ERE
1GWR        PDBfixfunc74  1GWR_A_fixed.pdb     EST_redock_1GWR
3UUD        PDBfixfunc74  3UUD_A_fixed.pdb     EST_redock_3UUD
6CBZ        PDBfixfunc74  6CBZ_A_fixed.pdb     EST_redock_6CBZ
3ERD        PDBfixfunc74  3ERD_A_fixed.pdb     DES_redock_3ERD
4ZN7        PDBfixfunc74  4ZN7_A_fixed.pdb     DES_redock_4ZN7
4MGC        PDBfixfunc74  4MGC_A_fixed.pdb     27M_redock_4MGC
4MG8        PDBfixfunc74  4MG8_A_fixed.pdb     27J_redock_4MG8
4TUZ        PDBfixfunc74  4TUZ_A_fixed.pdb     36J_redock_4TUZ
3UU7        PDBfixfunc74  3UU7_A_fixed.pdb     2OH_redock_3UU7
4MG9        PDBfixfunc74  4MG9_A_fixed.pdb     27K_redock_4MG9
4MGA        PDBfixfunc74  4MGA_A_fixed.pdb     27L_redock_4MGA
1G50        PDBfixfunc74  1G50_A_fixed.pdb     EST_redock_1G50


## Populate ideal ligand dictionary

In [12]:
idealliganddict = {}

In [ ]:
# This block assumes that these ligands are already present in the 'minimized_ideal_ligands' directory.
#minimized versions of ideal ligands
idealliganddict["27J_min"] = {'id':"27J_min", 'prep': "obabel-mmff94", 'localfilename': "27J_min.sdf"}
idealliganddict["27K_min"] = {'id':"27K_min", 'prep': "obabel-mmff94", 'localfilename': "27K_min.sdf"}
idealliganddict["27L_min"] = {'id':"27L_min", 'prep': "obabel-mmff94", 'localfilename': "27L_min.sdf"}

idealliganddict["27M_min"] = {'id':"27M_min", 'prep': "obabel-mmff94", 'localfilename': "27M_min.sdf"}
idealliganddict["2OH_min"] = {'id':"2OH_min", 'prep': "obabel-mmff94", 'localfilename': "2OH_min.sdf"}
idealliganddict["36J_min"] = {'id':"36J_min", 'prep': "obabel-mmff94", 'localfilename': "36J_min.sdf"}

idealliganddict["Caffeine_min"] = {'id':"Caffeine_min", 'prep': "obabel-mmff94", 'localfilename': "Caffeine_min.sdf"}
idealliganddict["DES_min"] = {'id':"DES_min", 'prep': "obabel-mmff94", 'localfilename': "DES_min.sdf"}
idealliganddict["EE2_min"] = {'id':"EE2_min", 'prep': "obabel-mmff94", 'localfilename': "EE2_min.sdf"}

idealliganddict["EST_min"] = {'id':"EST_min", 'prep': "obabel-mmff94", 'localfilename': "EST_min.sdf"}
idealliganddict["Melatonin_min"] = {'id':"Melatonin_min", 'prep': "obabel-mmff94", 'localfilename': "Melatonin_min.sdf"}
idealliganddict["Testosterone_min"] = {'id':"Testosterone_min", 'prep': "obabel-mmff94", 'localfilename': "Testosterone_min.sdf"}

### Add pubchem sourced ligands (optional)

In [ ]:
# Do we need to copy the pubchem .sdf files from the location where we processed them to the minimized_ideal_ligands directory?
# Probably only need to run this if/when we change or update the ideal ligands sourced via CAS from PubChem.
pubchemdir = "preprocessed/pubchemcas/"
sdfminimizedfilesdir = pubchemdir + "sdfminimizedfiles/stillypuy/"


#!cp -R {sdfminimizedfilesdir}* {idealdir}

In [13]:
# This block adds ideal ligands to the dictionary that were sourced from PubChem via CAS number by the 'acquire-minimize' notebook.
# These files need to be present in the minimized_ideal_ligands folder and have 'cas' in the filename.

import os
import re

for entry in os.scandir(idealdir):
    if entry.is_file() and entry.name.endswith(".sdf"):
        print ("I see file " + entry.name)
        id = "test" + entry.name
        match = re.search(r'^cas-(.*)\.sdf$', entry.name)
        if (match):
            id = match.group(1)       
            idealliganddict[id] = {'id':id, 'prep': "obabel-mmff94", 'localfilename': entry.name}
            print("     I added file " + entry.name)
            

I see file cas-107534-96-3_min.sdf
     I added file cas-107534-96-3_min.sdf
I see file cas-540-97-6_min.sdf
     I added file cas-540-97-6_min.sdf
I see file cas-93413-62-8_min.sdf
     I added file cas-93413-62-8_min.sdf
I see file cas-537-46-2_min.sdf
     I added file cas-537-46-2_min.sdf
I see file cas-3930-20-9_min.sdf
     I added file cas-3930-20-9_min.sdf
I see file cas-28721-07-5_min.sdf
     I added file cas-28721-07-5_min.sdf
I see file cas-144-83-2_min.sdf
     I added file cas-144-83-2_min.sdf
I see file cas-481-29-8_min.sdf
     I added file cas-481-29-8_min.sdf
I see file cas-72-33-3_min.sdf
     I added file cas-72-33-3_min.sdf
I see file cas-53-43-0_min.sdf
     I added file cas-53-43-0_min.sdf
I see file cas-84-17-3_min.sdf
     I added file cas-84-17-3_min.sdf
I see file cas-57-63-6_min.sdf
     I added file cas-57-63-6_min.sdf
I see file cas-5976-61-4_min.sdf
     I added file cas-5976-61-4_min.sdf
I see file cas-1222-05-5_min.sdf
     I added file cas-1222-05-5_mi

In [14]:
lines = reportdict(idealliganddict, ["id","prep","localfilename"])
print("\n".join(lines))
print("Number of ligands: " + str(len(idealliganddict)))

id               prep           localfilename          
107534-96-3_min  obabel-mmff94  cas-107534-96-3_min.sdf
540-97-6_min     obabel-mmff94  cas-540-97-6_min.sdf   
93413-62-8_min   obabel-mmff94  cas-93413-62-8_min.sdf 
537-46-2_min     obabel-mmff94  cas-537-46-2_min.sdf   
3930-20-9_min    obabel-mmff94  cas-3930-20-9_min.sdf  
28721-07-5_min   obabel-mmff94  cas-28721-07-5_min.sdf 
144-83-2_min     obabel-mmff94  cas-144-83-2_min.sdf   
481-29-8_min     obabel-mmff94  cas-481-29-8_min.sdf   
72-33-3_min      obabel-mmff94  cas-72-33-3_min.sdf    
53-43-0_min      obabel-mmff94  cas-53-43-0_min.sdf    
84-17-3_min      obabel-mmff94  cas-84-17-3_min.sdf    
57-63-6_min      obabel-mmff94  cas-57-63-6_min.sdf    
5976-61-4_min    obabel-mmff94  cas-5976-61-4_min.sdf  
1222-05-5_min    obabel-mmff94  cas-1222-05-5_min.sdf  
18684-55-4_min   obabel-mmff94  cas-18684-55-4_min.sdf 
85-68-7_min      obabel-mmff94  cas-85-68-7_min.sdf    
7432-28-2_min    obabel-mmff94  cas-7432-28-2_mi

## Examine proteins to determine what chains and ligands are present in the proteins. (Optional informational step) ##

In [ ]:
import gemmi

for key, value in proteindict.items():
    
    structure = gemmi.read_structure(prodir + value['localfilename_fixed'])
    #structure = gemmi.read_structure(chemfilesdir + "6O4w_rcbs.pdb")
    ligands = []
    
    for model in structure:
        for chain in model:
            for res in chain:
                if res.het_flag != ' ':  # hetero-residue
                    if res.name not in ("HOH", "WAT", "H2O"):
                        #if res.seqid.num == 604:
                        ligands.append((res.name, chain.name, res.seqid.num))

    print("protein: " + value['localfilename'])
    print(set(ligands))

In [ ]:
from Bio.PDB import PDBParser

for key, value in proteindict.items():
    
  parser = PDBParser(QUIET=True)
  structure = parser.get_structure("prot", prodir + value['localfilename_fixed'])
  #structure = parser.get_structure("prot", prodir + value['localfilename_fixed'])
    
  print("Protein: " + value['localfilename'])
    
  for model in structure:
    print(f"  Model {model.id}:")
    chain_ids = [chain.id for chain in model]
    print("    Chains:", ", ".join(chain_ids))


## Visualize Ligands

In [ ]:
from rdkit import Chem

ligandfile = ligdir + '2R6_ideal_PubChem.sdf'
ligandfileout = ligdir + '2R6_ideal_PubChem_NOH.sdf'

# Load SDF file (remove Hs on read - most efficient)
mol = Chem.MolFromMolFile(ligandfile, removeHs=True)

# Or if already loaded with Hs:
# mol = Chem.MolFromMolFile("ligand.sdf", removeHs=False)
# mol = Chem.RemoveHs(mol)

# Write H-free SDF
writer = Chem.SDWriter(ligandfileout)
writer.write(mol)
writer.close()

print(f"Atoms before: {Chem.MolFromMolFile(ligandfileout, removeHs=False).GetNumAtoms()}")
print(f"Atoms after:  {mol.GetNumAtoms()}")

In [ ]:
import nglview as nv
from rdkit import Chem

# From SDF
view = nv.show_structure_file(ligdir + '2R6_ideal_PubChem.sdf')
view.add_representation('ball+stick')
view.camera = 'orthographic'
view.center()
view

In [ ]:
view = nv.show_structure_file(ligdir + '4o09_final_ligand_2R6_A.pdb')
view.add_representation('ball+stick')
view.camera = 'orthographic'
view.center()
view

In [ ]:
view = nv.show_structure_file(ligdir + '2R6_redock_4o09_final_A_obabel.sdf')
view.add_representation('ball+stick')
view.camera = 'orthographic'
view.center()
view

In [ ]:
view.display(gui=True) 

## Define Gnina and dataframe functions

In [15]:
#Define the functions that call Gnina, parse the results files, and write those results to the dataframe.
from rdkit import Chem
import subprocess
number_of_modes = 1 #Report how many modes from each Gnina run?
#sort_by_cnnvs = True

def executegnina(proteinid,ligandid,boxid):
    protein = proteindict[proteinid]
    ligand = idealliganddict[ligandid]
    box = dockedliganddict[boxid]
    p = prodir + protein["localfilename_fixed"]
    l = idealdir + ligand["localfilename"]
    b = dockeddir + box["localfilename"]
    return callgnina(p,l,b)


def executegninadecoy(proteinid,boxid):
    protein = proteindict[proteinid]
    box = dockedliganddict[boxid]
    p = prodir + protein["localfilename_fixed"]
    l = decoydir + "decoyligand.sdf"
    b = dockeddir + box["localfilename"]
    return callgnina(p,l,b)

    
def callgnina(p,l,b):
    
    #!~/octoberproject/gnina -r "{p}" -l "{l}" --autobox_ligand "{b}" -o "{docked}" --log "{log}" --exhaustiveness=16 --num_modes=8 --seed 0 --pose_sort_order CNNaffinity --no_gpu  
    #!~/octoberproject/gnina -r "{p}" -l "{l}" --autobox_ligand "{b}" -o "{docked}" --log "{log}" --exhaustiveness=16 --num_modes=9 --seed 0 --pose_sort_order CNNscore --no_gpu  
    #!"{GNINA_LOC}" -r "{p}" -l "{l}" --autobox_ligand "{b}" -o "{docked}" --log "{log}" --exhaustiveness=16 --num_modes=9 --seed 0 --pose_sort_order CNNscore "{GNINA_PARAMETER}"

    if not GNINA_PARAMETER:
        gnina_cmd = [GNINA_LOC, "-r", p, "-l", l, "--autobox_ligand", b, "--autobox_add", "4", "-o", docked, "--log", log, 
       "--exhaustiveness=16", "--num_modes=9", "--seed", "0", "--pose_sort_order", "CNNscore"]

    else:
        gnina_cmd = [GNINA_LOC, "-r", p, "-l", l, "--autobox_ligand", b, "--autobox_add", "4", "-o", docked, "--log", log, 
       "--exhaustiveness=16", "--num_modes=9", "--seed", "0", "--pose_sort_order", "CNNscore", 
       GNINA_PARAMETER]

    #return True #To skip the call for Gnina 
    
    print("Call gnina:", " ".join(gnina_cmd))
 
    try:
        result = subprocess.run(gnina_cmd, check=True, capture_output=True, text=True)
        print("stdout:", result.stdout)
    except subprocess.CalledProcessError as e:
        # FAILURE (returncode != 0)
        print("Failure")
        print(f"Return code: {e.returncode}")
        print("stderr:", e.stderr)
        return False
    else:
        #only execute obrms if gnina ran successfully
        #Compute RMSD of two docked ligand files and save results to rmsdlog file.
        !obrms -f "{b}" "{docked}" | tee "{rmsdlog}"
        return True
    
    
def getdockedresultdf(docked):
    rmsddf = pd.read_csv(rmsdlog, sep=" ", header=None)
    rows = []
    for i, mol in enumerate(Chem.SDMolSupplier(docked)):
        if mol is None:
            continue
        rows.append({
            "pose": i,
            "CNNscore": float(mol.GetProp("CNNscore")),
            "CNN_VS": float(mol.GetProp("CNN_VS")),
            "CNNaffinity": float(mol.GetProp("CNNaffinity")),
            "RMSD": float(rmsddf.iloc[i,2]),
            "affinity": float(mol.GetProp("minimizedAffinity")),
        })
    df = pd.DataFrame(rows)
    if(sort_by_cnnvs):
        df.sort_values(by="CNN_VS", ascending=False, inplace=True)
    return(df)

def writeresulttodf(pro,lig,box):
    resultsdf = getdockedresultdf(docked)
    #for index, row in resultsdf.iterrows():
    for index, row in resultsdf.head(number_of_modes).iterrows():
        #write to the resultsdf here.
        temp = [pro, lig, box, "{:.4f}".format(row['CNNscore']), "{:.4f}".format(row['CNN_VS']), "{:.4f}".format(row['RMSD']), "{:.4f}".format(row['affinity'])]
        outputdf.loc[len(outputdf)] = temp


In [ ]:
resultsdf = getdockedresultdf(docked)
print("resultsdf:")
print(resultsdf)

## Run the docking simulations

In [16]:
# Establish new empty dataframe
# Or set it to empty for the next new run.

import pandas as pd
outputdf = pd.DataFrame(columns = ["protein", "ideal_ligand", "native_ligand", "CNN_pose", "CNN_VS", "RMSD", "affinity"])

In [ ]:
#Test block. Dock just one pair.
pkey = "1ERE"
ikey = "27K_min"
dockedid = "EST_redock_1ERE"
gninasuccess = executegnina(pkey, ikey, dockedid)
if (gninasuccess):
    writeresulttodf(pkey, ikey, dockedid) 

In [17]:
# Iterate though protein dictionary and ligand dictionary and perform Gnina docking for every combo.
from itertools import islice

pro = proteindict
#pro = dict(islice(proteindict.items(), 3))
ide = idealliganddict
#ide = dict(islice(idealliganddict.items(), 3))

rundecoys = False

start = time.time()

#iterate through proteins
for pkey, pvalue in pro.items():
    #iterate through ideal ligands
    for ikey, ivalue in ide.items():
        print(f"{pkey} meets {ikey} at {pvalue['dockedid']}")
        executegnina(pkey, ikey, pvalue['dockedid'])
        writeresulttodf(pkey,ikey,pvalue['dockedid'])   

    #iterate through decoy ligands if dataframe exists and is populated
    if (rundecoys):
        for idx, row in decoydf.head(2).iterrows():
        #for idx, row in decoydf.iterrows():
            writeonedecoytoFS(row)
            print(f"{pkey} meets {row['ID']} at {pvalue['dockedid']}") 
            executegninadecoy(pkey, pvalue['dockedid'])
            writeresulttodf(pkey, row['ID'], pvalue['dockedid'])  

end = time.time()
elapsed = end - start
print(f"Gnina runs completed in {format_time(elapsed)}")  # 00:01:05

1ERE meets 107534-96-3_min at EST_redock_1ERE
Call gnina: gnina -r preprocessed/proteinprep/1ERE_A_fixed.pdb -l preprocessed/minimized_ideal_ligand/stillypuy/cas-107534-96-3_min.sdf --autobox_ligand preprocessed/dockedligands/EST_redock_1ERE_A.sdf --autobox_add 4 -o results/docked.sdf --log results/gninalog.txt --exhaustiveness=16 --num_modes=9 --seed 0 --pose_sort_order CNNscore
stdout:               _             
             (_)            
   __ _ _ __  _ _ __   __ _ 
  / _` | '_ \| | '_ \ / _` |
 | (_| | | | | | | | | (_| |
  \__, |_| |_|_|_| |_|\__,_|
   __/ |                    
  |___/                     

gnina  master:6567d54+   Built Jan 22 2026.
gnina is based on smina and AutoDock Vina.
Please cite appropriately.

Commandline: gnina -r preprocessed/proteinprep/1ERE_A_fixed.pdb -l preprocessed/minimized_ideal_ligand/stillypuy/cas-107534-96-3_min.sdf --autobox_ligand preprocessed/dockedligands/EST_redock_1ERE_A.sdf --autobox_add 4 -o results/docked.sdf --log results/gninal

[12:20:47] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[12:20:47] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[12:20:47] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[12:20:47] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[12:20:47] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[12:20:47] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[12:20:47] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[12:20:47] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[12:20:47] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol

stdout:               _             
             (_)            
   __ _ _ __  _ _ __   __ _ 
  / _` | '_ \| | '_ \ / _` |
 | (_| | | | | | | | | (_| |
  \__, |_| |_|_|_| |_|\__,_|
   __/ |                    
  |___/                     

gnina  master:6567d54+   Built Jan 22 2026.
gnina is based on smina and AutoDock Vina.
Please cite appropriately.

Commandline: gnina -r preprocessed/proteinprep/1ERE_A_fixed.pdb -l preprocessed/minimized_ideal_ligand/stillypuy/cas-540-97-6_min.sdf --autobox_ligand preprocessed/dockedligands/EST_redock_1ERE_A.sdf --autobox_add 4 -o results/docked.sdf --log results/gninalog.txt --exhaustiveness=16 --num_modes=9 --seed 0 --pose_sort_order CNNscore
Using random seed: 0

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
10911 | pose 0 | initial pose not within box

mode |  affinity  |  intramol  |    CNN     |   CNN
     | (kcal/mol) | (kcal/mol)

[12:20:50] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[12:20:50] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[12:20:50] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[12:20:50] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[12:20:50] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[12:20:50] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[12:20:50] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[12:20:50] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[12:20:50] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol

stdout:               _             
             (_)            
   __ _ _ __  _ _ __   __ _ 
  / _` | '_ \| | '_ \ / _` |
 | (_| | | | | | | | | (_| |
  \__, |_| |_|_|_| |_|\__,_|
   __/ |                    
  |___/                     

gnina  master:6567d54+   Built Jan 22 2026.
gnina is based on smina and AutoDock Vina.
Please cite appropriately.

Commandline: gnina -r preprocessed/proteinprep/1ERE_A_fixed.pdb -l preprocessed/minimized_ideal_ligand/stillypuy/cas-93413-62-8_min.sdf --autobox_ligand preprocessed/dockedligands/EST_redock_1ERE_A.sdf --autobox_add 4 -o results/docked.sdf --log results/gninalog.txt --exhaustiveness=16 --num_modes=9 --seed 0 --pose_sort_order CNNscore
Using random seed: 0

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
125017 | pose 0 | initial pose not within box

mode |  affinity  |  intramol  |    CNN     |   CNN
     | (kcal/mol) | (kcal/m

...

[13:33:26] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:33:26] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:33:26] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:33:26] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:33:26] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:33:26] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:33:26] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:33:26] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:33:26] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol

stdout:               _             
             (_)            
   __ _ _ __  _ _ __   __ _ 
  / _` | '_ \| | '_ \ / _` |
 | (_| | | | | | | | | (_| |
  \__, |_| |_|_|_| |_|\__,_|
   __/ |                    
  |___/                     

gnina  master:6567d54+   Built Jan 22 2026.
gnina is based on smina and AutoDock Vina.
Please cite appropriately.

Commandline: gnina -r preprocessed/proteinprep/1G50_A_fixed.pdb -l preprocessed/minimized_ideal_ligand/stillypuy/cas-125-71-3_min.sdf --autobox_ligand preprocessed/dockedligands/EST_redock_1G50_A.sdf --autobox_add 4 -o results/docked.sdf --log results/gninalog.txt --exhaustiveness=16 --num_modes=9 --seed 0 --pose_sort_order CNNscore
Using random seed: 0

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
5360696 | pose 0 | initial pose not within box

mode |  affinity  |  intramol  |    CNN     |   CNN
     | (kcal/mol) | (kcal/mo

[13:33:30] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:33:30] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:33:30] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:33:30] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:33:30] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:33:30] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:33:30] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:33:30] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:33:30] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol

stdout:               _             
             (_)            
   __ _ _ __  _ _ __   __ _ 
  / _` | '_ \| | '_ \ / _` |
 | (_| | | | | | | | | (_| |
  \__, |_| |_|_|_| |_|\__,_|
   __/ |                    
  |___/                     

gnina  master:6567d54+   Built Jan 22 2026.
gnina is based on smina and AutoDock Vina.
Please cite appropriately.

Commandline: gnina -r preprocessed/proteinprep/1G50_A_fixed.pdb -l preprocessed/minimized_ideal_ligand/stillypuy/cas-34816-55-2_min.sdf --autobox_ligand preprocessed/dockedligands/EST_redock_1G50_A.sdf --autobox_add 4 -o results/docked.sdf --log results/gninalog.txt --exhaustiveness=16 --num_modes=9 --seed 0 --pose_sort_order CNNscore
Using random seed: 0

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
11954041 | pose 0 | initial pose not within box
11954041 | pose 0 | ligand outside box

mode |  affinity  |  intramol  |    CN

[13:33:35] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:33:35] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:33:35] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:33:35] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:33:35] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:33:35] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:33:35] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:33:35] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:33:35] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol

In [18]:
print(f"Gnina runs completed in {format_time(elapsed)}")  # 00:01:05

Gnina runs completed in 01:12:57


In [19]:
# Preview results
print(outputdf)

    protein     ideal_ligand    native_ligand CNN_pose  CNN_VS    RMSD  \
0      1ERE  107534-96-3_min  EST_redock_1ERE   0.4440  3.1107     inf   
1      1ERE     540-97-6_min  EST_redock_1ERE   0.3896  1.6835     inf   
2      1ERE   93413-62-8_min  EST_redock_1ERE   0.8530  6.2981     inf   
3      1ERE     537-46-2_min  EST_redock_1ERE   0.5510  2.2265     inf   
4      1ERE    3930-20-9_min  EST_redock_1ERE   0.3190  1.9835     inf   
..      ...              ...              ...      ...     ...     ...   
879    1G50  104746-04-5_min  EST_redock_1G50   0.5996  3.5010     inf   
880    1G50      84-16-2_min  EST_redock_1G50   0.9594  7.9554     inf   
881    1G50    2163-68-0_min  EST_redock_1G50   0.4861  2.6263     inf   
882    1G50     125-71-3_min  EST_redock_1G50   0.2005  1.1843     inf   
883    1G50   34816-55-2_min  EST_redock_1G50   0.8775  7.1261  0.8520   

    affinity  
0    -8.1018  
1    19.5746  
2    -8.6876  
3    -5.9379  
4    -6.7322  
..       ...  
879  -

## Order, rearrance, output as .csv

In [ ]:
# Order by LIGAND, then CNN_pose desc. Change order of columns.
outputdfsorted = pd.DataFrame(columns = ["protein", "ideal_ligand", "native_ligand", "CNN_pose", "CNN_VS", "RMSD"])
for group_name, group_df in outputdf.groupby("ideal_ligand"):
    oneliganddf = group_df.sort_values(by="CNN_pose", ascending=False)
    outputdfsorted = pd.concat([outputdfsorted, pd.DataFrame(oneliganddf)], ignore_index=True)
    
outputdfsorted = outputdfsorted[["ideal_ligand", "protein", "native_ligand", "CNN_pose", "CNN_VS", "RMSD"]]

print(outputdfsorted)

In [21]:
# Order by PROTEIN, then CNN_pose desc. Change order of columns.
outputdfsorted = pd.DataFrame(columns = ["protein", "ideal_ligand", "native_ligand", "CNN_pose", "CNN_VS", "RMSD"])
for group_name, group_df in outputdf.groupby("protein"):
    oneliganddf = group_df.sort_values(by="CNN_pose", ascending=False)
    outputdfsorted = pd.concat([outputdfsorted, pd.DataFrame(oneliganddf)], ignore_index=True)
    
outputdfsorted = outputdfsorted[["protein", "ideal_ligand", "native_ligand", "CNN_pose", "CNN_VS", "RMSD"]]

print(outputdfsorted)

    protein     ideal_ligand    native_ligand CNN_pose  CNN_VS    RMSD
0      1ERE      53-16-7_min  EST_redock_1ERE   0.9894  8.1667  0.6404
1      1ERE     474-86-2_min  EST_redock_1ERE   0.9836  7.9732  0.7425
2      1ERE      50-28-2_min  EST_redock_1ERE   0.9824  8.1945  0.5143
3      1ERE      57-91-0_min  EST_redock_1ERE   0.9797  8.0580  0.7362
4      1ERE    5696-58-2_min  EST_redock_1ERE   0.9762  7.4867  0.7178
..      ...              ...              ...      ...     ...     ...
879    6CBZ     540-97-6_min  EST_redock_6CBZ   0.3415  1.4555     inf
880    6CBZ   66722-44-9_min  EST_redock_6CBZ   0.3289  2.0692     inf
881    6CBZ     125-71-3_min  EST_redock_6CBZ   0.2095  1.3233     inf
882    6CBZ  134523-00-5_min  EST_redock_6CBZ   0.1748  1.2158     inf
883    6CBZ    7432-28-2_min  EST_redock_6CBZ   0.1382  0.8470     inf

[884 rows x 6 columns]


In [20]:
# Write to .csv
import csv
import time
epoch = int(time.time())
outfile = resultsdir + "results" + str(epoch) + ".csv"
outputdfsorted.to_csv(outfile, mode='w', index=False, header=True)

NameError: name 'outputdfsorted' is not defined